# Belgium VAR — CSV Direct Collector v2.1.1

市镇核心表批量采集；复用v2前测缓存；不启动浏览器。

非表格入口只保留默认CSV快照，未批量遍历的维度明确列入报告。缺少筛选元数据的入口不被解释为完整年度面板。

空间范围检查是诊断，不是源站所有单元格完整性的证明。缺失、未知、抑制和合计行保留原样，不补零、不相加。

本修订尚未在制作环境执行测试；以下步骤包含本机语法检查后的基础自检、定向联网前测和覆盖报告。

本版新增独立的 heterogeneity supplemental continuation 单元格：自动复用缓存、探测 worker-count indicator，并只续采缺失的 nationality/origin/generation worker-count 数据。

## 1. 原有路径与缓存配置
不要更改OUTPUT_DIR，才能复用v2缓存。

In [1]:

from pathlib import Path
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("请使用 Python 3.10 或更新版本。")

PROJECT_DIR = Path(r"D:\OneDrive - Universiteit Utrecht\31_BL_Network")

# 只读取此前已经生成的目录，不需要旧浏览器采集器继续运行。
OLD_DIR = PROJECT_DIR / "raw_data" / "VAR_all_dimensions"

# 新版独立输出目录，不覆盖旧数据。
OUTPUT_DIR = PROJECT_DIR / "raw_data" / "VAR_CSV_Direct_v2"

CONFIG = {
    "years": list(range(2015, 2025)),
    "timeout_seconds": 240,
    "request_gap_seconds": 2.0,
    "max_retries": 3,
    "max_tasks": 200000,

    # expanded:
    #   尽量展开CSV中可核验的分类轴，其余有效控制使用笛卡尔积。
    #   只保留源站实际返回的记录，不推算未公开联合数据。
    #
    # marginals:
    #   年份/年龄/身份等核心控制交叉，
    #   其余未能展开的控制逐维变化，减少请求数量。
    "mode": "expanded",

    # 地图导出的CSV也列入采集；不默默缩减为仅表格视图。
    "include_map_views": True,

    # 只检查已有的元数据文件，不读取旧数据分页或浏览器缓存。
    "max_metadata_file_mb": 30,

    # 未找到其他维度时，不猜测其取值。
    # OD旧代码已使用过的参数仅作为联网待核验候选。
    "use_od_fallback_candidates": True,

    # 每次请求都核对已经确认有CSV标签列的单值筛选。
    "verify_returned_labels": True,

    # 原始文件哈希验证。
    "verify_cached_hashes": True,

    # 元数据中的列取值示例上限，不限制实际下载行数。
    "max_distinct_values_per_column": 1200,

    # URL长度过长时停止该请求，不截短筛选条件。
    "max_url_length": 16000,

    # 连续网络失败达到阈值后暂停；重新运行可续采。
    "stop_after_consecutive_errors": 3,

    # 每个参数前测最多选择3个不同值。
    "probe_values_per_control": 3,

    # 若源站需要特定参数拼写，可在检查报告后按精确原名设置。
    # 格式：{"parameter::原名": "实际URL字段"}
    "url_field_overrides": {},
}

for folder in (
    "metadata", "metadata/requests", "metadata/source",
    "raw", "reports", "quarantine"
):
    (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("旧目录，只读：", OLD_DIR)
print("新版输出目录：", OUTPUT_DIR)
print("模式：", CONFIG["mode"])

旧目录，只读： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_all_dimensions
新版输出目录： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_CSV_Direct_v2
模式： expanded


## 2. 原CSV下载器
保留原传输版本及缓存键，不重新运行全目录发现。

In [2]:

import csv
import gzip
import hashlib
import io
import itertools
import json
import math
import os
import re
import shutil
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import zipfile

from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

VERSION = "2.0.0"
HOST = "analytics.data.wewis.vlaanderen.be"
BASE = f"https://{HOST}"
WORKBOOK_PREFIX = "/t/Steunpuntwerk/views/VARpt2-T3/"
CREDIT = "Steunpunt Werk - Vlaamse Arbeidsrekening o.b.v. DWH AM&SB - KSZ, BISA"

STATIC_VIEWS = {
    "intro", "bnw - overzicht", "pendel - overzicht", "bw - overzicht"
}

# 这三个标签对应关系来自此前OD导出的实际字段。
# 其他字段优先使用CSV中与筛选名称完全对应的列。
KNOWN_COLUMNS = {
    "Geslacht": ["Geslacht_titel", "Geslacht"],
    "EDU2": ["EDU_titel", "EDU2"],
    "WSE42_naam": ["Sector_titel", "WSE42_naam"],
}

CORE_ROLES = {
    "year", "age", "employment_status",
    "geography_level", "indicator", "layout"
}

class AccessDenied(RuntimeError):
    pass

class ExportError(RuntimeError):
    pass

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def normal(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode().casefold()
    return re.sub(r"\s+", " ", text).strip()

def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return {"__nonfinite__": str(value)}
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value

def jd(value):
    return json.dumps(
        json_safe(value), ensure_ascii=False,
        sort_keys=True, separators=(",", ":"), allow_nan=False
    )

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(
        json_safe(value), ensure_ascii=False, indent=2, allow_nan=False
    )
    temp = path.with_name(path.name + ".part")
    temp.write_text(text, encoding="utf-8")
    os.replace(temp, path)

def read_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8-sig"),
        parse_constant=lambda x: {"__nonfinite__": x}
    )

def write_report(path, rows):
    rows = list(rows)
    fields = list(dict.fromkeys(k for row in rows for k in row))
    if not fields:
        fields = ["note"]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".part")
    with temp.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                k: jd(v) if isinstance(v, (dict, list, tuple)) else v
                for k, v in row.items()
            })
    os.replace(temp, path)

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def module_of(name):
    name = normal(name)
    if name.startswith("bnw"):
        return "residence"
    if name.startswith("bw"):
        return "workplace"
    if "pendel" in name:
        return "commuting"
    return "other"

def role_of(name):
    name = normal(name)
    patterns = [
        ("year", r"jaar|year|annee"),
        ("age", r"leeftijd|age.?group"),
        ("education", r"onderwijs|opleid|geschool|education|^edu"),
        ("sex", r"geslacht|gender|^sex"),
        ("sector", r"sector|wse|nace|industry"),
        ("employment_status", r"statuut|socio.?econom|arbeidsmarktpositie"),
        ("geography_level", r"geograf.*niveau|regio.*niveau|schaalniveau"),
        ("geography", r"woonplaats|werkplaats|gemeente|provincie|gewest|^nis"),
        ("nationality", r"nationalit"),
        ("origin", r"herkomst|generatie|origin"),
        ("work_regime", r"arbeidsregime|prestatietype|voltijd|deeltijd"),
        ("establishment_size", r"grootte|size"),
        ("indicator", r"indicator|maatstaf|meetwaarde|measure.?name|variabele"),
        ("layout", r"dimensie|uitsplits|rijen|kolommen|^rij$|^kolom$"),
    ]
    return next((role for role, p in patterns if re.search(p, name)), "other")

def scalar(value):
    if isinstance(value, dict) or isinstance(value, (list, tuple)):
        return None
    if value is None:
        return None
    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        if value.is_integer():
            return str(int(value))
    if isinstance(value, bool):
        return "true" if value else "false"
    return str(value)

def domain_item(value, kind):
    if not isinstance(value, dict):
        value = {"value": value, "formattedValue": scalar(value)}

    order = (
        ["value", "nativeValue", "aliasValue", "formattedValue"]
        if kind == "parameter"
        else ["aliasValue", "formattedValue", "value", "nativeValue"]
    )

    labels = list(dict.fromkeys(
        s for k in ("value", "nativeValue", "formattedValue", "aliasValue")
        if (s := scalar(value.get(k))) is not None
    ))
    send = next(
        (s for k in order if (s := scalar(value.get(k))) is not None),
        None
    )
    return None if send is None else {"send": send, "labels": labels}

def finite_domain(control):
    kind = control.get("kind", "filter")
    raw = control.get("values") or []

    if not raw and kind == "parameter" and control.get("domainType") == "range":
        a = domain_item(control.get("minValue"), kind)
        b = domain_item(control.get("maxValue"), kind)
        step = control.get("stepSize")
        if step is None and control.get("dataType") in ("int", "integer"):
            step = 1
        try:
            lo, hi, increment = float(a["send"]), float(b["send"]), float(step)
            n = int(math.floor((hi - lo) / increment + 1e-8)) + 1
            if (
                all(math.isfinite(x) for x in (lo, hi, increment))
                and increment > 0 and 0 < n <= 10000
            ):
                raw = [lo + i * increment for i in range(n)]
        except (TypeError, ValueError, KeyError, ZeroDivisionError, OverflowError):
            raw = []

    result = {}
    for value in raw:
        item = domain_item(value, kind)
        if item:
            result[item["send"]] = item
    return list(result.values())

def walk(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk(child)

def csv_url(source):
    if not source:
        raise ValueError("发布目录没有提供真实URL。")

    u = urllib.parse.urlsplit(urllib.parse.urljoin(BASE, str(source)))
    path = u.path

    # 支持官方浏览地址中的 #/site/.../views/... 形式。
    if "/views/" in u.fragment:
        fragment = urllib.parse.urlsplit(u.fragment)
        path = re.sub(r"^/site/([^/]+)/views/", r"/t/\1/views/", fragment.path)

    if u.hostname != HOST or not path.startswith(WORKBOOK_PREFIX):
        raise ValueError(f"不属于目标公开工作簿：{source}")

    path = re.sub(r"\.(csv|png|pdf|xlsx)$", "", path, flags=re.I)
    path = urllib.parse.quote(urllib.parse.unquote(path), safe="/:@-")
    return urllib.parse.urlunsplit(("https", HOST, path + ".csv", "", ""))

def import_catalogue():
    metadata_dir = OLD_DIR / "metadata"
    if not metadata_dir.is_dir():
        raise FileNotFoundError(
            f"没有找到旧元数据目录：{metadata_dir}\n"
            "这版需要已生成的 workbook.json / catalog.partial.json，"
            "但不需要重新运行浏览器发现。"
        )

    files = sorted(
        (
            p for p in metadata_dir.rglob("*.json")
            if "requests" not in p.parts
            and "source" not in p.parts
            and "plan" not in p.name.lower()
            and p.stat().st_size <= CONFIG["max_metadata_file_mb"] * 1024**2
        ),
        key=lambda p: p.stat().st_mtime
    )

    sheets, entries, issues, consumed = {}, {}, [], []

    for path in files:
        try:
            obj = read_json(path)
        except Exception as exc:
            issues.append({"file": str(path), "issue": "metadata_read_error", "detail": str(exc)})
            continue

        useful = False
        for node in walk(obj):
            for key in ("sheets", "publishedSheetsInfo"):
                values = node.get(key)
                if isinstance(values, list):
                    for s in values:
                        if isinstance(s, dict) and s.get("name") and s.get("url"):
                            sheets[s["name"]] = dict(s)
                            useful = True

            if node.get("view") and node.get("worksheet"):
                entries[(node["view"], node["worksheet"])] = dict(node)
                useful = True

            # 兼容按视图保存的 inventory 结果。
            if node.get("view") and isinstance(node.get("worksheets"), list):
                for w in node["worksheets"]:
                    if isinstance(w, dict) and w.get("name"):
                        e = {
                            "view": node["view"],
                            "worksheet": w["name"],
                            "columns": w.get("columns", []),
                            "filters": w.get("filters", []),
                            "parameters": node.get("parameters", []),
                        }
                        entries[(e["view"], e["worksheet"])] = e
                        useful = True

        if useful:
            digest = sha_file(path)
            target = OUTPUT_DIR / "metadata/source" / f"{digest[:12]}_{path.name}"
            if not target.exists():
                shutil.copy2(path, target)
            consumed.append({"path": str(path), "sha256": digest})

    # 已有采集器可能只在 entries 中保存视图URL。
    for e in entries.values():
        if e.get("view_url") and e["view"] not in sheets:
            sheets[e["view"]] = {"name": e["view"], "url": e["view_url"]}

    if not sheets:
        raise RuntimeError(
            "没有读取到带真实URL的发布目录。"
            "请检查旧目录中的 metadata/workbook.json；"
            "代码不会通过删除空格等方式猜测所有视图地址。"
        )

    views = []
    for name, sheet in sheets.items():
        contexts = [e for e in entries.values() if e["view"] == name]
        static = normal(name) in STATIC_VIEWS
        is_map = "(kaart)" in normal(name) or "pendel - k -" in normal(name)
        try:
            endpoint = csv_url(sheet["url"])
            error = None
        except Exception as exc:
            endpoint, error = None, str(exc)

        views.append({
            "name": name,
            "module": module_of(name),
            "csv_url": endpoint,
            "worksheet_names": sorted({e["worksheet"] for e in contexts}),
            "known_worksheet_count": len({e["worksheet"] for e in contexts}),
            "contexts": contexts,
            "static": static,
            "is_map": is_map,
            "url_error": error,
        })

    # 先处理统计表，地图与其他视图在后；没有从目录删除它们。
    views.sort(key=lambda v: (
        v["static"], "tabel" not in normal(v["name"]), v["module"], v["name"]
    ))

    present = {v["module"] for v in views}
    absent = {"residence", "workplace", "commuting"} - present
    if absent:
        raise RuntimeError(f"发布目录缺少模块：{sorted(absent)}。不会用OD目录冒充全部模块。")

    write_json(OUTPUT_DIR / "metadata/source_manifest.json", consumed)
    write_json(OUTPUT_DIR / "metadata/imported_catalogue.json", views)
    write_report(OUTPUT_DIR / "reports/import_issues.csv", issues)
    write_report(
        OUTPUT_DIR / "reports/view_inventory.csv",
        [{k: x for k, x in v.items() if k != "contexts"} for v in views]
    )
    return views

def controls_for(view):
    found = {}
    for e in view["contexts"]:
        for kind, key in (("parameter", "parameters"), ("filter", "filters")):
            for raw in e.get(key, []):
                if not isinstance(raw, dict) or not raw.get("name"):
                    continue
                c = {**raw, "kind": kind}
                name = c["name"]
                identifier = f"{kind}::{name}"
                values = finite_domain(c)

                # 非分类筛选没有适用的通用CSV多值表达，不擅自转换。
                if kind == "filter" and c.get("filterType") not in (None, "categorical"):
                    values = []

                if identifier not in found:
                    found[identifier] = {
                        "key": identifier, "name": name, "kind": kind,
                        "role": role_of(name), "values": [],
                        "metadata_available": True,
                    }
                merged = {x["send"]: x for x in found[identifier]["values"]}
                merged.update({x["send"]: x for x in values})
                found[identifier]["values"] = list(merged.values())

    # 只对Pendel候选补充此前代码使用过的参数；仍必须通过新前测。
    if CONFIG["use_od_fallback_candidates"] and view["module"] == "commuting":
        fallback = {
            "jaar": [str(y) for y in CONFIG["years"]],
            "Leeftijdsklasse": ["15-24", "25-54", "55-64", "15-64", "18-64", "20-64", "15-70"],
            "Statuut": ["Werkend", "Loontrekkend", "Zelfstandig"],
            "Geslacht": ["Totaal", "Mannen", "Vrouwen", "Onbekend"],
            "EDU2": ["Totaal", "Kortgeschoold", "Middengeschoold", "Hooggeschoold", "Onbekend"],
        }
        names = {c["name"] for c in found.values()}
        for name, values in fallback.items():
            if name not in names:
                key = f"filter::{name}"
                found[key] = {
                    "key": key, "name": name, "kind": "filter",
                    "role": role_of(name),
                    "values": [{"send": x, "labels": [x]} for x in values],
                    "metadata_available": False,
                }

    for c in found.values():
        if c["role"] == "year":
            wanted = {str(y) for y in CONFIG["years"]}
            c["values"] = [
                x for x in c["values"]
                if wanted.intersection(x["labels"] + [x["send"]])
            ]
    return list(found.values())

def label(value, column):
    value = " ".join(str(value).strip(" ,\t\r\n").split())
    if column == "Geslacht_titel" and value == "Mannen en vrouwen":
        value = "Totaal"
    if column in ("EDU_titel", "Sector_titel") and not value:
        value = "Totaal"
    return normal(value)

def mapped_column(control, header):
    candidates = [control["name"]] + KNOWN_COLUMNS.get(control["name"], [])
    exact = [x for x in header if x in candidates]
    return exact[0] if len(set(exact)) == 1 else None

def inspect_csv(path):
    with gzip.open(path, "rb") as f:
        head = f.read(65536)

    encoding = "utf-16" if head.startswith((b"\xff\xfe", b"\xfe\xff")) else "utf-8-sig"
    text = head.decode(encoding, errors="replace").lstrip()

    if not text:
        return {
            "rows": 0, "header": [], "values": {},
            "truncated_columns": [], "sample": [],
            "fingerprint": "empty", "encoding": encoding, "delimiter": None
        }

    if text.startswith(("<", "{", "[")):
        raise ExportError("收到HTML/JSON或错误页，不能作为CSV数据。")

    try:
        delimiter = csv.Sniffer().sniff(text, delimiters=",;\t").delimiter
    except csv.Error:
        first = text.splitlines()[0]
        delimiter = "\t" if "\t" in first else ","

    sets, truncated, sample = {}, set(), []
    n, checksum = 0, 0
    limit = CONFIG["max_distinct_values_per_column"]

    with gzip.open(path, "rt", encoding=encoding, errors="strict", newline="") as f:
        reader = csv.reader(f, delimiter=delimiter, strict=True)
        header = next(reader, [])

        if not header:
            raise ExportError("非空响应没有可读表头。")

        for i, col in enumerate(header):
            sets[str(i)] = set()

        for row in reader:
            if not row:
                continue
            if len(row) != len(header):
                raise ExportError(
                    f"CSV列数不一致：表头{len(header)}列，第{n + 2}行{len(row)}列。"
                )
            n += 1
            if len(sample) < 3:
                sample.append(row)

            # 行序无关指纹，用于对照请求；不把顺序变化误认成筛选有效。
            h = hashlib.sha256(jd(row).encode("utf-8")).digest()
            checksum = (checksum + int.from_bytes(h, "big")) % (1 << 256)

            for i, value in enumerate(row):
                key = str(i)
                if value in sets[key]:
                    continue
                if len(sets[key]) < limit:
                    sets[key].add(value)
                else:
                    truncated.add(key)

    return {
        "rows": n,
        "header": header,
        "values": {k: sorted(v) for k, v in sets.items()},
        "truncated_columns": sorted(truncated),
        "sample": sample,
        "fingerprint": f"{jd(header)}:{n}:{checksum:064x}",
        "encoding": encoding,
        "delimiter": delimiter,
    }

class DirectCSV:
    def __init__(self):
        self.last_request = 0.0

    def url(self, view, parameters):
        if not view.get("csv_url"):
            raise ExportError("视图没有经过目录确认的CSV地址。")
        query = {
            ":showVizHome": "no",
            ":embed": "y",
            ":isGuestRedirectFromVizportal": "y",
            **{str(k): str(v) for k, v in parameters.items()},
        }
        url = view["csv_url"] + "?" + urllib.parse.urlencode(sorted(query.items()))
        if len(url) > CONFIG["max_url_length"]:
            raise ExportError(f"URL过长：{len(url)}字符；没有截短参数。")
        return url

    def fetch(self, view, parameters, purpose):
        url = self.url(view, parameters)
        rid = hashlib.sha256(url.encode("utf-8")).hexdigest()[:28]
        raw = OUTPUT_DIR / "raw" / view["module"] / f"{rid}.csv.gz"
        meta_path = OUTPUT_DIR / "metadata/requests" / f"{rid}.json"
        raw.parent.mkdir(parents=True, exist_ok=True)

        if meta_path.exists() and raw.exists():
            try:
                m = read_json(meta_path)
                valid = (
                    m.get("status") == "received"
                    and m.get("version") == VERSION
                    and m.get("url") == url
                    and m.get("file_bytes") == raw.stat().st_size
                )
                if valid and CONFIG["verify_cached_hashes"]:
                    valid = sha_file(raw) == m["file_sha256"]
                if valid:
                    return m
            except (OSError, ValueError, KeyError):
                pass

        temp = raw.with_name(raw.name + ".part")
        last_error = None

        for attempt in range(CONFIG["max_retries"] + 1):
            wait = CONFIG["request_gap_seconds"] - (time.monotonic() - self.last_request)
            if wait > 0:
                time.sleep(wait)
            self.last_request = time.monotonic()

            try:
                req = urllib.request.Request(
                    url,
                    headers={
                        "User-Agent": "VAR-public-CSV-research-collector/" + VERSION,
                        "Accept": "text/csv",
                        "Accept-Encoding": "identity",
                    }
                )
                with urllib.request.urlopen(
                    req, timeout=CONFIG["timeout_seconds"]
                ) as response:
                    final_url = response.geturl()
                    if urllib.parse.urlsplit(final_url).hostname != HOST:
                        raise AccessDenied(f"请求被重定向至其他站点：{final_url}")

                    ctype = response.headers.get("Content-Type", "").lower()
                    encoding = response.headers.get("Content-Encoding", "").lower()
                    if encoding not in ("", "identity"):
                        raise ExportError(f"收到未预期的HTTP内容编码：{encoding}")

                    if "html" in ctype or "json" in ctype:
                        body = response.read(32768)
                        q = OUTPUT_DIR / "quarantine" / f"{rid}.response"
                        q.write_bytes(body)
                        raise ExportError(f"预期CSV，实际Content-Type={ctype}")

                    total = 0
                    content_hash = hashlib.sha256()
                    expected = response.headers.get("Content-Length")

                    with gzip.open(temp, "wb", compresslevel=6) as output:
                        while True:
                            block = response.read(1024 * 1024)
                            if not block:
                                break
                            output.write(block)
                            total += len(block)
                            content_hash.update(block)

                    if expected is not None and total != int(expected):
                        raise OSError(f"响应未读完整：{total}/{expected}字节")

                stats = inspect_csv(temp)
                os.replace(temp, raw)
                record = {
                    "version": VERSION, "request_id": rid,
                    "view": view["name"], "module": view["module"],
                    "url": url, "final_url": final_url,
                    "requested_parameters": parameters,
                    "purpose_first_seen": purpose,
                    "downloaded_utc": utc_now(), "status": "received",
                    "raw_file": raw.relative_to(OUTPUT_DIR).as_posix(),
                    "response_bytes": total,
                    "response_sha256": content_hash.hexdigest(),
                    "file_bytes": raw.stat().st_size,
                    "file_sha256": sha_file(raw),
                    "source_credit": CREDIT,
                    **stats,
                }
                write_json(meta_path, record)
                return record

            except urllib.error.HTTPError as exc:
                last_error = f"HTTP {exc.code}: {exc.reason}"
                if exc.code in (401, 403):
                    raise AccessDenied(last_error) from exc
                if exc.code not in (408, 425, 429, 500, 502, 503, 504):
                    break
                retry_after = exc.headers.get("Retry-After", "")
                delay = 0.0
                try:
                    delay = float(retry_after)
                except ValueError:
                    try:
                        delay = max(
                            0.0,
                            parsedate_to_datetime(retry_after).timestamp() - time.time()
                        )
                    except (TypeError, ValueError, OverflowError):
                        pass
                if delay:
                    time.sleep(delay)

            except AccessDenied:
                raise

            except ExportError as exc:
                last_error = str(exc)
                if temp.exists():
                    quarantine = OUTPUT_DIR / "quarantine" / f"{rid}.csv.gz"
                    os.replace(temp, quarantine)
                break

            except (OSError, ValueError, csv.Error, urllib.error.URLError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"

            finally:
                temp.unlink(missing_ok=True)

            if attempt < CONFIG["max_retries"]:
                time.sleep(min(60, 2 ** (attempt + 2)))

        result = {
            "version": VERSION, "request_id": rid,
            "view": view["name"], "module": view["module"],
            "url": url, "requested_parameters": parameters,
            "status": "failed", "error": last_error,
            "checked_utc": utc_now(),
        }
        write_json(meta_path, result)
        return result

CLIENT = DirectCSV()

def observed(record, column):
    if column not in record.get("header", []):
        return None
    indices = [i for i, name in enumerate(record["header"]) if name == column]
    if len(indices) != 1:
        return None
    key = str(indices[0])
    if key in record.get("truncated_columns", []):
        return None
    return {label(x, column) for x in record["values"].get(key, [])}

def labels_match(record, column, item):
    values = observed(record, column)
    expected = {label(x, column) for x in item["labels"] + [item["send"]]}
    return bool(values) and values.issubset(expected)

def probe_values(control):
    values = control["values"]
    if not values:
        return []
    # 总量/未知之外的类别优先，减少两个对照都空的概率。
    preferred = [
        v for v in values
        if normal(v["send"]) not in ("totaal", "onbekend", "all")
    ]
    pool = preferred or values
    positions = [0, len(pool) - 1, len(pool) // 2]
    selected = {}
    for pos in positions:
        selected[pool[pos]["send"]] = pool[pos]
    return list(selected.values())[:CONFIG["probe_values_per_control"]]

def probe_catalogue(views):
    profiles, rows = [], []
    profile_path = OUTPUT_DIR / "metadata/profiles.json"

    # 参数核验进度也支持续跑。
    previous = {}
    if profile_path.exists():
        previous = {p["name"]: p for p in read_json(profile_path)}

    for number, view in enumerate(views, 1):
        print(f"[CSV probe {number}/{len(views)}] {view['name']}", flush=True)

        signature = hashlib.sha256(
            jd({
                "version": VERSION,
                "view": view,
                "years": CONFIG["years"],
                "overrides": CONFIG["url_field_overrides"],
                "include_map_views": CONFIG["include_map_views"],
            }).encode("utf-8")
        ).hexdigest()

        old = previous.get(view["name"])
        if old and old.get("signature") == signature and old.get("probe_finished"):
            profiles.append(old)
            rows.extend(old.get("control_report", []))
            print("  [cached] 已保存的前测", flush=True)
            continue

        p = {
            "name": view["name"], "module": view["module"],
            "csv_url": view["csv_url"], "signature": signature,
            "active_controls": [], "control_report": [],
            "known_worksheet_count": view["known_worksheet_count"],
            "worksheet_names": view["worksheet_names"],
            "probe_finished": False,
        }

        if view["static"]:
            p.update(status="static_navigation_view", probe_finished=True)
        elif view["is_map"] and not CONFIG["include_map_views"]:
            p.update(status="explicitly_excluded_map", probe_finished=True)
        elif view["url_error"]:
            p.update(status="missing_url", error=view["url_error"], probe_finished=True)
        else:
            baseline = CLIENT.fetch(view, {}, "baseline")
            p["baseline_request_id"] = baseline["request_id"]

            if baseline["status"] != "received":
                p.update(status="baseline_failed", error=baseline.get("error"))
            else:
                p["baseline_rows"] = baseline["rows"]
                controls = controls_for(view)
                p["metadata_controls_count"] = len(controls)

                for c in controls:
                    report = {
                        "view": view["name"], "control": c["key"],
                        "role": c["role"], "domain_size": len(c["values"]),
                        "metadata_available": c["metadata_available"],
                    }

                    samples = probe_values(c)
                    if not samples:
                        report["result"] = "no_finite_domain_unresolved"
                        p["control_report"].append(report)
                        continue

                    override = CONFIG["url_field_overrides"].get(c["key"])
                    keys = [override] if override else [c["name"]]
                    if not override and c["kind"] == "parameter":
                        keys.append("Parameters." + c["name"])

                    accepted = None
                    attempts = []

                    for key in dict.fromkeys(keys):
                        tests = [
                            (item, CLIENT.fetch(view, {key: item["send"]}, "field_probe"))
                            for item in samples
                        ]
                        good = [
                            (item, result) for item, result in tests
                            if result["status"] == "received" and result["rows"] > 0
                        ]

                        column = (
                            mapped_column(c, good[0][1]["header"])
                            if good else None
                        )
                        checked = (
                            column is not None
                            and bool(good)
                            and all(labels_match(r, column, item) for item, r in good)
                        )

                        # 只有返回标签没有明确冲突时，才考虑敏感性证据。
                        contradicted = (
                            column is not None
                            and any(not labels_match(r, column, item) for item, r in good)
                        )
                        sensitive = (
                            len(good) >= 2
                            and len({r["fingerprint"] for _, r in good}) >= 2
                        )

                        attempts.append({
                            "url_field": key,
                            "request_ids": [r["request_id"] for _, r in tests],
                            "rows": [r.get("rows") for _, r in tests],
                            "label_column": column,
                            "label_checked": checked,
                            "sensitive": sensitive,
                            "contradicted": contradicted,
                        })

                        if checked or (sensitive and not contradicted):
                            accepted = {
                                **c, "url_field": key,
                                "label_column": column if checked else None,
                                "evidence": "label_checked" if checked else "sensitivity_only",
                                "expansion_value": None,
                            }

                            # 只对分类筛选尝试展开，参数不会用空值冒充All。
                            if c["kind"] == "filter" and checked and c["role"] not in CORE_ROLES:
                                all_values = [x["send"] for x in c["values"]]
                                candidates = [""]

                                if (
                                    all("," not in x for x in all_values)
                                    and len(",".join(all_values)) < 7000
                                ):
                                    candidates.append(",".join(all_values))

                                needed = set()
                                for _, r in good:
                                    needed.update(observed(r, column) or set())

                                for expansion in dict.fromkeys(candidates):
                                    expanded = CLIENT.fetch(
                                        view, {key: expansion}, "expansion_probe"
                                    )
                                    actual = observed(expanded, column)
                                    if (
                                        expanded["status"] == "received"
                                        and actual is not None
                                        and len(actual) >= 2
                                        and needed.issubset(actual)
                                    ):
                                        accepted["expansion_value"] = expansion
                                        accepted["expansion_evidence"] = (
                                            "includes_nonempty_probe_labels;"
                                            "not_a_proof_of_all_source_categories"
                                        )
                                        break
                            break

                    report["attempts"] = attempts
                    if accepted:
                        p["active_controls"].append(accepted)
                        report.update(
                            result=accepted["evidence"],
                            url_field=accepted["url_field"],
                            label_column=accepted["label_column"],
                            expanded=accepted["expansion_value"] is not None,
                        )
                    else:
                        report["result"] = "unverified_not_used_for_bulk_labelling"

                    p["control_report"].append(report)
                    print(
                        "  ", c["key"], "->", report["result"],
                        "| values:", len(c["values"]), flush=True
                    )

                    # 每个控制完成后持久保存，保留联网证据。
                    write_json(profile_path, profiles + [p])

                p.update(status="probed", probe_finished=True)

        profiles.append(p)
        rows.extend(p.get("control_report", []))
        write_json(profile_path, profiles)
        write_report(OUTPUT_DIR / "reports/control_validation.csv", rows)

    return profiles

def make_plan(profiles):
    specifications, issues = [], []
    total_upper_bound = 0

    for p in profiles:
        if p.get("status") != "probed":
            issues.append({"view": p["name"], "issue": p.get("status")})
            continue

        active = p["active_controls"]
        names = [c["url_field"] for c in active]
        if len(set(names)) != len(names):
            issues.append({"view": p["name"], "issue": "duplicate_url_field_requires_review"})
            continue

        fixed, axes = {}, []
        for c in active:
            if c["expansion_value"] is not None:
                fixed[c["url_field"]] = c["expansion_value"]
            else:
                axes.append(c)

        if CONFIG["mode"] == "expanded":
            estimate = math.prod(len(c["values"]) for c in axes)
        elif CONFIG["mode"] == "marginals":
            core = [c for c in axes if c["role"] in CORE_ROLES]
            extra = [c for c in axes if c["role"] not in CORE_ROLES]
            estimate = (
                math.prod(len(c["values"]) for c in core)
                * (1 + sum(len(c["values"]) for c in extra))
            )
        else:
            raise ValueError("mode必须为expanded或marginals。")

        total_upper_bound += estimate
        specifications.append({"profile": p, "fixed": fixed, "axes": axes, "estimate": estimate})

        if p["known_worksheet_count"] != 1:
            issues.append({
                "view": p["name"],
                "issue": "dashboard_internal_worksheet_coverage_unresolved",
                "known_worksheet_count": p["known_worksheet_count"],
                "worksheet_names": p["worksheet_names"],
            })

        for r in p.get("control_report", []):
            if r["result"] not in ("label_checked", "sensitivity_only"):
                issues.append({
                    "view": p["name"], "control": r["control"],
                    "issue": r["result"],
                })

        if not any(c["role"] == "year" for c in active):
            issues.append({
                "view": p["name"],
                "issue": "no_verified_year_control_default_export_only_or_other_axes"
            })

    write_report(OUTPUT_DIR / "reports/scope_issues.csv", issues)
    write_report(
        OUTPUT_DIR / "reports/plan_size_by_view.csv",
        [{
            "view": s["profile"]["name"],
            "estimated_queries": s["estimate"],
            "expanded_fields": list(s["fixed"]),
            "iterated_fields": [c["url_field"] for c in s["axes"]],
        } for s in specifications]
    )

    if total_upper_bound > CONFIG["max_tasks"]:
        raise RuntimeError(
            f"计划上限约{total_upper_bound:,}个请求，超过max_tasks="
            f"{CONFIG['max_tasks']:,}。\n"
            "尚未启动批量下载。请先检查reports/plan_size_by_view.csv。"
            "提高预算或改用marginals都会在配置中留下记录；不会静默截断。"
        )

    plan = OUTPUT_DIR / "metadata/plan.jsonl"
    temp = plan.with_name(plan.name + ".part")
    seen = set()

    with temp.open("w", encoding="utf-8") as f:
        for spec in specifications:
            p, axes, fixed = spec["profile"], spec["axes"], spec["fixed"]

            if CONFIG["mode"] == "expanded":
                combinations = (
                    list(zip(axes, values))
                    for values in itertools.product(*(c["values"] for c in axes))
                )
            else:
                core = [c for c in axes if c["role"] in CORE_ROLES]
                extra = [c for c in axes if c["role"] not in CORE_ROLES]

                def marginal_combinations(core=core, extra=extra):
                    for values in itertools.product(*(c["values"] for c in core)):
                        base = list(zip(core, values))
                        yield base
                        for c in extra:
                            for value in c["values"]:
                                yield base + [(c, value)]

                combinations = marginal_combinations()

            for combination in combinations:
                params = dict(fixed)
                checks = []
                evidence = []

                for c, value in combination:
                    params[c["url_field"]] = value["send"]
                    evidence.append({
                        "field": c["url_field"],
                        "requested": value["send"],
                        "evidence": c["evidence"],
                    })
                    if c["label_column"]:
                        checks.append({
                            "column": c["label_column"],
                            "item": value,
                        })

                view = {
                    "name": p["name"], "module": p["module"],
                    "csv_url": p["csv_url"],
                }
                url = CLIENT.url(view, params)
                rid = hashlib.sha256(url.encode("utf-8")).hexdigest()[:28]
                if rid in seen:
                    continue
                seen.add(rid)
                f.write(jd({
                    "request_id": rid,
                    "view": view,
                    "parameters": params,
                    "checks": checks,
                    "context_evidence": evidence,
                    "mode": CONFIG["mode"],
                }) + "\n")

    os.replace(temp, plan)
    plan_hash = sha_file(plan)
    write_json(OUTPUT_DIR / "metadata/plan_manifest.json", {
        "version": VERSION, "created_utc": utc_now(),
        "task_count": len(seen), "plan_sha256": plan_hash,
        "config": CONFIG,
        "scope_note": (
            "All tasks in this explicit CSV plan. "
            "Not a certification of every internal worksheet or every "
            "unpublished/cross-classified source cell."
        )
    })
    print("计划请求数：", f"{len(seen):,}")
    print("范围问题清单：", OUTPUT_DIR / "reports/scope_issues.csv")
    return plan

def iter_plan():
    with (OUTPUT_DIR / "metadata/plan.jsonl").open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def run_collection(max_new_requests=None):
    manifest = read_json(OUTPUT_DIR / "metadata/plan_manifest.json")
    plan = OUTPUT_DIR / "metadata/plan.jsonl"
    if sha_file(plan) != manifest["plan_sha256"]:
        raise RuntimeError("计划文件发生变化，请重新运行make_plan。")

    results = {}
    status_path = OUTPUT_DIR / "reports/task_status.json"
    if status_path.exists():
        results = read_json(status_path)

    consecutive = 0
    processed = 0

    try:
        for number, task in enumerate(iter_plan(), 1):
            if max_new_requests is not None and processed >= max_new_requests:
                print("达到本轮任务上限。剩余任务仍为pending。")
                break

            record = CLIENT.fetch(task["view"], task["parameters"], "collection")
            processed += 1

            if record["status"] != "received":
                status = "failed"
                consecutive += 1
                detail = record.get("error")
            else:
                consecutive = 0
                bad = []
                if CONFIG["verify_returned_labels"] and record["rows"] > 0:
                    bad = [
                        c["column"] for c in task["checks"]
                        if not labels_match(record, c["column"], c["item"])
                    ]

                if bad:
                    status = "label_mismatch"
                    detail = {"columns": bad}
                elif record["rows"] == 0:
                    status = "empty_response_not_zero"
                    detail = None
                else:
                    status = "received"
                    detail = None

            results[task["request_id"]] = {
                "request_id": task["request_id"],
                "view": task["view"]["name"],
                "module": task["view"]["module"],
                "status": status,
                "rows": record.get("rows"),
                "detail": detail,
                "checked_utc": utc_now(),
            }
            write_json(status_path, results)

            print(
                f"[{number}/{manifest['task_count']}] "
                f"{task['view']['name']} | {status} | "
                f"rows={record.get('rows', '?')}",
                flush=True
            )

            if consecutive >= CONFIG["stop_after_consecutive_errors"]:
                print("连续请求失败，暂停本轮。重新运行本单元格可以续采。")
                break

    except KeyboardInterrupt:
        print("已暂停，已完成文件保留。")

    finally:
        audit_collection()

def audit_collection():
    status_path = OUTPUT_DIR / "reports/task_status.json"
    statuses = read_json(status_path) if status_path.exists() else {}
    rows = []

    for task in iter_plan():
        row = statuses.get(task["request_id"], {
            "request_id": task["request_id"],
            "view": task["view"]["name"],
            "module": task["view"]["module"],
            "status": "pending",
        })
        rows.append(row)

    counts = Counter(row["status"] for row in rows)
    finished = counts["received"] + counts["empty_response_not_zero"]

    summary = {
        "checked_utc": utc_now(),
        "planned_queries": len(rows),
        "task_status_counts": dict(counts),
        "all_planned_queries_received": finished == len(rows) and bool(rows),
        "full_source_coverage_certified": False,
        "notes": [
            "请求完成与全源数据完整是不同概念。",
            "empty_response_not_zero表示没有返回记录，不能解释为人数为零。",
            "scope_issues.csv保留未核验控制和多工作表导出的覆盖问题。",
            "请求年份/年龄/身份保存在requested_parameters，不伪装成源CSV自带字段。",
            "合计、未知、抑制标记、原始数值字符串均保留。",
        ],
    }

    write_report(OUTPUT_DIR / "reports/coverage.csv", rows)
    write_json(OUTPUT_DIR / "reports/summary.json", summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

def archive_results(include_raw=True):
    archive = OUTPUT_DIR.parent / (
        "VAR_CSV_for_synthesis_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip"
    )
    folders = ["metadata", "reports"]
    if include_raw:
        folders.append("raw")

    with zipfile.ZipFile(
        archive, "w", compression=zipfile.ZIP_DEFLATED,
        compresslevel=1, allowZip64=True
    ) as z:
        for folder in folders:
            for path in sorted((OUTPUT_DIR / folder).rglob("*")):
                if path.is_file() and not path.name.endswith(".part"):
                    z.write(path, path.relative_to(OUTPUT_DIR).as_posix())
    print("输出压缩包：", archive)
    return archive

def self_test():
    assert scalar(float("nan")) is None
    assert json_safe(float("nan")) == {"__nonfinite__": "nan"}
    assert json.loads(jd({"n": float("inf")}))["n"]["__nonfinite__"] == "inf"
    assert module_of("BNW - T1 (tabel)") == "residence"
    assert module_of("BW - T1 (tabel)") == "workplace"
    assert module_of("Pendel Uit - Tabel") == "commuting"
    assert label("Mannen en vrouwen", "Geslacht_titel") == "totaal"
    assert label("", "EDU_titel") == "totaal"

    u = csv_url(BASE + WORKBOOK_PREFIX + "PendelUit-Tabel?:iid=123")
    assert u.endswith("/PendelUit-Tabel.csv") and ":iid" not in u

    control = {
        "kind": "parameter", "domainType": "range", "dataType": "integer",
        "minValue": {"value": 2017}, "maxValue": {"value": 2024},
        "stepSize": 1,
    }
    assert len(finite_domain(control)) == 8
    assert domain_item({"value": 0, "formattedValue": "0"}, "parameter")["send"] == "0"
    print("本机基础自检通过；这不替代联网前测。")

In [3]:
self_test()

本机基础自检通过；这不替代联网前测。


## 3. 加载优化规则

In [4]:

from copy import deepcopy

PLANNER_VERSION = "2.1.0"
TARGET_GEOGRAPHY = "Gemeente"

# 这是识别明显地区限制的诊断阈值，不是完整性证明。
MIN_MUNICIPALITY_SHARE = 0.95

NON_MUNICIPAL_LABELS = {
    "null", "%null%", "totaal", "onbekend", "andere",
    "internationaal", "duitsland", "frankrijk",
    "groothertogdom luxemburg", "nederland", "belgie",
}

def is_table_view(p):
    return "tabel" in normal(p["name"])

def is_location_filter(c):
    """地点选择器，不作为彼此独立的笛卡尔积维度。"""
    return c.get("kind") == "filter" and bool(re.search(
        r"^(gemeente|gewest|provincie|referentieregio)(_|$)|^geo_map",
        normal(c["name"])
    ))

def is_level_control(c):
    """空间层级选择器，与具体地点选择器分开。"""
    return normal(c["name"]) in {
        "regio", "geografisch niveau", "geo", "geo 1", "geo_jv"
    }

def choose_seed(c):
    """仅用于少量前测，不缩减正式计划中的类别。"""
    values = c["values"]
    if not values:
        raise ValueError(f"空分类域：{c['key']}")

    if c.get("role") == "year":
        def year_key(v):
            candidates = [
                int(str(x)) for x in v.get("labels", []) + [v["send"]]
                if re.fullmatch(r"\d{4}", str(x))
            ]
            return max(candidates or [-1])
        return max(values, key=year_key)

    preferred = (
        ["20-64", "15-64", "15-70", "Totaal"]
        if c.get("role") == "age"
        else ["Werkend", "Totaal", "Loontrekkend", "Loontrekkenden"]
    )
    for target in preferred:
        for v in values:
            labels = v.get("labels", []) + [v["send"]]
            if normal(target) in {normal(x) for x in labels}:
                return v
    return values[0]

def seed_parameters(controls):
    params = {}
    for c in controls:
        field = c["url_field"]
        if c.get("expansion_value") is not None:
            params[field] = c["expansion_value"]
        elif len(c.get("values", [])) == 1:
            params[field] = c["values"][0]["send"]
        elif c.get("role") in {"year", "age", "employment_status"}:
            params[field] = choose_seed(c)["send"]
    return params

def contains_known_categories(record, c):
    """要求每个已知类别至少有一个匹配的返回标签。"""
    column = c.get("label_column")
    if (
        record.get("status") != "received"
        or not record.get("rows")
        or not column
    ):
        return False

    actual = observed(record, column)
    if actual is None:
        return False

    return all(
        any(
            label(x, column) in actual
            for x in item.get("labels", []) + [item["send"]]
        )
        for item in c["values"]
    )

def municipality_names(controls, exact_name=None):
    result = set()
    for c in controls:
        if exact_name is not None:
            selected = c["name"] == exact_name
        else:
            selected = normal(c["name"]).startswith("gemeente")
        if selected:
            for item in c.get("values", []):
                value = normal(str(item["send"]).strip())
                if value and value not in NON_MUNICIPAL_LABELS:
                    result.add(value)
    return result

def geography_score(record, source_profile, global_names):
    """按实际返回标签检查地理范围；不依据总行数推断。"""
    if record.get("status") != "received" or not record.get("rows"):
        return 0.0, {"error": record.get("error", "empty_response")}

    details = []
    controls = source_profile.get("active_controls", [])

    if source_profile["module"] == "commuting":
        targets = [
            ("Gemeente_woon", "Woonplaats"),
            ("Gemeente_werk", "Werkplaats"),
        ]
        for field, column in targets:
            expected = municipality_names(controls, field) or global_names
            actual = observed(record, column) or set()
            matched = expected & actual
            details.append({
                "column": column,
                "matched": len(matched),
                "reference_names": len(expected),
                "share": len(matched) / len(expected) if expected else 0.0,
                "names_not_returned": sorted(expected - actual),
            })
    else:
        expected = municipality_names(controls) or global_names
        if not expected:
            return 0.0, {"error": "no_municipality_reference_names"}

        candidates = []
        for column in record.get("header", []):
            actual = observed(record, column)
            if actual is not None:
                candidates.append((len(expected & actual), column, actual))

        if not candidates:
            return 0.0, {"error": "no_inspectable_geographic_label_column"}

        count, column, actual = max(candidates, key=lambda x: x[0])
        details.append({
            "column": column,
            "matched": count,
            "reference_names": len(expected),
            "share": count / len(expected),
            "names_not_returned": sorted(expected - actual),
        })

    return min(d["share"] for d in details), {"columns": details}

def planner_note(p, key, result):
    p.setdefault("control_report", []).append({
        "view": p["name"],
        "control": "planner::" + key,
        "role": "planner_policy",
        "domain_size": 0,
        "metadata_available": False,
        "result": result,
    })

def prepare_optimized_profiles(profiles):
    """
    不改写原profiles.json。
    地理筛选不再相乘；市镇核心表保留正式分类组合。
    非表格入口仅保留默认快照，并明确记录未完成维度遍历。
    """
    global_names = set()
    for p in profiles:
        global_names.update(municipality_names(p.get("active_controls", [])))

    if len(global_names) < 100:
        raise RuntimeError(
            "没有足够的市镇名称参考，停止自动地理范围校验。"
        )

    prepared, policy_rows, checks = [], [], []
    requested_years = {str(y) for y in CONFIG["years"]}

    for index, source in enumerate(profiles, 1):
        q = deepcopy(source)
        print(f"[prepare {index}/{len(profiles)}] {q['name']}", flush=True)

        if q.get("status") != "probed":
            prepared.append(q)
            continue

        if not is_table_view(q) or not q.get("active_controls"):
            reason = (
                "default_snapshot_only_missing_dimension_metadata"
                if is_table_view(q)
                else "auxiliary_snapshot_only_dimension_sweep_deferred"
            )
            q["active_controls"] = []
            planner_note(q, "scope", reason)
            policy_rows.append({"view": q["name"], "policy": reason})
            prepared.append(q)
            continue

        locations = [
            c for c in q["active_controls"] if is_location_filter(c)
        ]
        controls = [
            c for c in q["active_controls"] if not is_location_filter(c)
        ]

        blocked = None
        for c in controls:
            if is_level_control(c):
                selected = [
                    v for v in c["values"]
                    if normal(TARGET_GEOGRAPHY) in {
                        normal(x) for x in v.get("labels", []) + [v["send"]]
                    }
                ]
                if len(selected) != 1:
                    blocked = f"cannot_select_unique_municipality_level::{c['key']}"
                    break
                c["values"] = selected
                c["expansion_value"] = None

            if c.get("role") == "year":
                c["values"] = [
                    v for v in c["values"]
                    if requested_years.intersection(
                        str(x) for x in v.get("labels", []) + [v["send"]]
                    )
                ]
                if not c["values"]:
                    blocked = "no_requested_year_in_profile"
                    break

        if blocked:
            q["status"] = "blocked_optimized_scope"
            planner_note(q, "scope", blocked)
            prepared.append(q)
            policy_rows.append({"view": q["name"], "policy": blocked})
            continue

        q["active_controls"] = controls
        seed = seed_parameters(controls)

        # 先检查不传具体地点筛选的响应；
        # 必要时再检查将这些地点筛选清空的响应。
        options = [("published_default_locations", {})]
        if locations:
            options.append((
                "empty_location_filters",
                {c["url_field"]: "" for c in locations}
            ))

        best = None
        for name, location_params in options:
            record = CLIENT.fetch(
                q, {**seed, **location_params}, "optimized_geography_probe"
            )
            score, detail = geography_score(record, source, global_names)
            checks.append({
                "view": q["name"],
                "stage": "geography",
                "policy": name,
                "request_id": record["request_id"],
                "score": score,
                "details": detail,
            })
            if best is None or score > best[0]:
                best = (score, name, location_params)

            # 所有参考市镇标签均已出现，不再增加清空筛选请求。
            if score == 1.0:
                break

        if best is None or best[0] < MIN_MUNICIPALITY_SHARE:
            q["status"] = "blocked_geographic_scope"
            planner_note(q, "geography", "broad_municipal_scope_not_confirmed")
            prepared.append(q)
            policy_rows.append({
                "view": q["name"],
                "policy": "blocked_geographic_scope",
                "best_score": best[0] if best else None,
            })
            print("  [blocked] 市镇范围校验未通过；保留诊断。", flush=True)
            continue

        if best[1] == "empty_location_filters":
            for c in locations:
                c["expansion_value"] = ""
                c["expansion_evidence"] = (
                    "geographic_scope_probe;"
                    "not_a_certification_of_every_source_cell"
                )
                controls.append(c)

        # 只有返回了全部已知类别，才展开有标签列的年龄/身份。
        # 年份仍逐年请求，以控制单次响应大小。
        for c in list(controls):
            if (
                c.get("role") not in {"age", "employment_status"}
                or c.get("kind") != "filter"
                or c.get("evidence") != "label_checked"
                or not c.get("label_column")
                or c.get("expansion_value") is not None
            ):
                continue

            parameters = seed_parameters(controls)
            parameters[c["url_field"]] = ""

            record = CLIENT.fetch(
                q, parameters, "optimized_core_expansion_probe"
            )
            already_expanded = [
                x for x in controls
                if x.get("role") in {"age", "employment_status"}
                and x.get("expansion_value") is not None
            ]
            passed = all(
                contains_known_categories(record, x)
                for x in already_expanded + [c]
            )

            checks.append({
                "view": q["name"],
                "stage": "core_expansion",
                "field": c["url_field"],
                "request_id": record["request_id"],
                "passed": passed,
                "rows": record.get("rows"),
            })
            if passed:
                c["expansion_value"] = ""
                c["optimized_core_expansion"] = True
                c["expansion_evidence"] = (
                    "all_known_category_labels_present_in_seed_probe;"
                    "other_contexts_require_coverage_audit"
                )
                print("  [expand]", c["url_field"], flush=True)

        q["active_controls"] = controls
        planner_note(
            q, "geography",
            "municipality_scope;location_selectors_not_cartesian_axes"
        )
        policy_rows.append({
            "view": q["name"],
            "policy": "municipal_table",
            "location_policy": best[1],
            "geographic_score": best[0],
            "iterated_fields": [
                c["url_field"] for c in controls
                if c.get("expansion_value") is None
            ],
            "expanded_fields": [
                c["url_field"] for c in controls
                if c.get("expansion_value") is not None
            ],
        })
        prepared.append(q)

        write_json(OUTPUT_DIR / "reports/optimized_preflight.json", checks)
        write_report(OUTPUT_DIR / "reports/optimized_view_policy.csv", policy_rows)

    write_json(OUTPUT_DIR / "metadata/profiles_optimized.json", prepared)
    write_json(OUTPUT_DIR / "reports/optimized_preflight.json", checks)
    write_report(OUTPUT_DIR / "reports/optimized_view_policy.csv", policy_rows)
    return prepared

def save_optimized_plan(prepared):
    # 原下载器版本不变，避免仅因版本号改变而失效已有缓存。
    # 只备份将被更新的计划/报告，不移动raw或requests缓存。
    backup = OUTPUT_DIR / "_planner_backups" / datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    for relative in (
        "metadata/plan.jsonl",
        "metadata/plan_manifest.json",
        "reports/plan_size_by_view.csv",
        "reports/scope_issues.csv",
    ):
        path = OUTPUT_DIR / relative
        if path.exists():
            destination = backup / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, destination)

    CONFIG["mode"] = "expanded"
    path = make_plan(prepared)

    manifest_path = OUTPUT_DIR / "metadata/plan_manifest.json"
    manifest = read_json(manifest_path)
    manifest["planner_version"] = PLANNER_VERSION
    manifest["scope_note"] = (
        "Municipal table collection plus explicitly marked default snapshots. "
        "Non-table dimension sweeps and missing metadata remain unresolved. "
        "Not a certification of complete VAR source coverage."
    )
    write_json(manifest_path, manifest)
    return path

def audit_core_expansions(prepared):
    """检查其他年份/条件下的实际返回类别，不将缺席类别补零。"""
    by_name = {p["name"]: p for p in prepared}
    rows = []
    for task in iter_plan():
        path = (
            OUTPUT_DIR / "metadata/requests"
            / (task["request_id"] + ".json")
        )
        if not path.exists():
            continue
        record = read_json(path)
        if record.get("status") != "received":
            continue

        p = by_name[task["view"]["name"]]
        for c in p.get("active_controls", []):
            if c.get("optimized_core_expansion"):
                rows.append({
                    "request_id": task["request_id"],
                    "view": p["name"],
                    "field": c["url_field"],
                    "rows": record.get("rows"),
                    "all_known_labels_returned": contains_known_categories(record, c),
                    "note": (
                        "Missing labels may be absent, suppressed, inapplicable, "
                        "or affected by other filters; not zero observations."
                    ),
                })
    write_report(
        OUTPUT_DIR / "reports/expanded_core_category_coverage.csv", rows
    )
    print("分类覆盖报告：",
          OUTPUT_DIR / "reports/expanded_core_category_coverage.csv")


## 4. 读取已完成的前测记录
本单元格不联网。

In [5]:
PROFILES = read_json(OUTPUT_DIR / 'metadata/profiles.json')
print('前测视图数：', len(PROFILES))

前测视图数： 48


## 5. 定向检查地理范围及年龄/身份展开
只补充这些检查，不重做全部字段发现。地理检查失败的表会标记并保留诊断，不混入已验证的市镇批次。

In [6]:
PREPARED = prepare_optimized_profiles(PROFILES)

[prepare 1/48] Pendel In - Tabel
[prepare 2/48] Pendel Uit - Tabel
[prepare 3/48] BNW - T1 (tabel)
  [expand] Leeftijdsklasse
[prepare 4/48] BNW - T2 (tabel)
  [expand] Leeftijdsklasse
[prepare 5/48] BNW - T3 (tabel)
  [expand] Leeftijdsklasse
[prepare 6/48] BNW - T4 (tabel)
[prepare 7/48] BNW - T5ab (tabel)
  [expand] Leeftijdsklasse
[prepare 8/48] BNW - T5cd (tabel)
  [expand] Leeftijdsklasse
[prepare 9/48] BNW - T6ab (tabel) 
  [expand] Leeftijdsklasse
[prepare 10/48] BNW - T6cd (tabel)
  [expand] Leeftijdsklasse
[prepare 11/48] BW - JV (tabel)
[prepare 12/48] BW - T1 (tabel)
[prepare 13/48] BW - T2a (tabel)
  [expand] Leeftijdsklasse
[prepare 14/48] BW - T2b (tabel)
[prepare 15/48] BW - T3a herk (tabel)
  [expand] Leeftijdsklasse
[prepare 16/48] BW - T3a nat (tabel)
  [expand] Leeftijdsklasse
[prepare 17/48] BW - T3b herk (tabel)
[prepare 18/48] BW - T3b nat (tabel)
[prepare 19/48] Pendel - K - In - Intens
[prepare 20/48] Pendel - K - In - Stroom - Gem
[prepare 21/48] Pendel - K - 

## 6. 生成新计划
先备份旧规划报告，再生成任务。预算检查仍保留，不会通过静默截断来降低请求数。

请查看reports/optimized_view_policy.csv，区分核心表、默认快照与未通过检查的入口。

In [7]:
PLAN_PATH = save_optimized_plan(PREPARED)

计划请求数： 7,118
范围问题清单： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_CSV_Direct_v2\reports\scope_issues.csv


## 7. 批量采集
原始CSV和前测缓存继续复用。中断后重新运行本单元格续采。不要同时用两个内核运行同一输出目录。

In [9]:
run_collection()

[1/7118] Pendel In - Tabel | received | rows=1100224
[2/7118] Pendel In - Tabel | received | rows=1098325
[3/7118] Pendel In - Tabel | received | rows=1092819
[4/7118] Pendel In - Tabel | received | rows=1110791
[5/7118] Pendel In - Tabel | received | rows=190283
[6/7118] Pendel In - Tabel | received | rows=918193
[7/7118] Pendel In - Tabel | received | rows=411455
[8/7118] Pendel In - Tabel | received | rows=1098370
[9/7118] Pendel In - Tabel | received | rows=1095365
[10/7118] Pendel In - Tabel | received | rows=1091292
[11/7118] Pendel In - Tabel | received | rows=1108130
[12/7118] Pendel In - Tabel | received | rows=191490
[13/7118] Pendel In - Tabel | received | rows=920422
[14/7118] Pendel In - Tabel | received | rows=407533
[15/7118] Pendel In - Tabel | received | rows=1094227
[16/7118] Pendel In - Tabel | received | rows=1091811
[17/7118] Pendel In - Tabel | received | rows=1087464
[18/7118] Pendel In - Tabel | received | rows=1103062
[19/7118] Pendel In - Tabel | received | ro

## 8. 检查任务覆盖与新增展开字段的类别覆盖

In [10]:
SUMMARY = audit_collection()
audit_core_expansions(PREPARED)

{
  "checked_utc": "2026-09-22T10:48:33.129540+00:00",
  "planned_queries": 7118,
  "task_status_counts": {
    "received": 6600,
    "empty_response_not_zero": 518
  },
  "all_planned_queries_received": true,
  "full_source_coverage_certified": false,
  "notes": [
    "请求完成与全源数据完整是不同概念。",
    "empty_response_not_zero表示没有返回记录，不能解释为人数为零。",
    "scope_issues.csv保留未核验控制和多工作表导出的覆盖问题。",
    "请求年份/年龄/身份保存在requested_parameters，不伪装成源CSV自带字段。",
    "合计、未知、抑制标记、原始数值字符串均保留。"
  ]
}
分类覆盖报告： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_CSV_Direct_v2\reports\expanded_core_category_coverage.csv


## 9. 异质性补采：自动检查并续采缺失的 worker-count 指标

本单元格不会改写原 7,118-task final plan。它先扫描现有 request cache，复用已经存在且通过验证的文件；只有在 2017–2024、20–64 岁的 BNW-T5ab / T6ab / T6cd 中缺少 **Aantal werkenden (n)** 时才联网补采。

补采对象：resident nationality、origin/herkomst、generation。输出独立的 `metadata/supplemental_heterogeneity_plan.jsonl` 与 `reports/supplemental_heterogeneity_*`，便于后续 master-data notebook 明确合并，不把补采任务伪装成原 final plan。


In [ ]:

# ============================================================
# Supplemental heterogeneity continuation collector
# - scans cache first
# - probes the Tableau indicator only when needed
# - downloads only missing 2017-2024 / age 20-64 worker-count files
# - keeps the original 7,118-task final plan unchanged
# ============================================================

SUPPLEMENT_NAME = "heterogeneity_worker_counts_v1"
SUPPLEMENT_YEARS = list(range(2017, 2025))
SUPPLEMENT_AGE = "20-64"
SUPPLEMENT_MEASURE_HINTS = ("aantal", "werkend")

SUPPLEMENT_SPECS = [
    {
        "key": "resident_nationality",
        "view": "BNW - T5ab (tabel)",
        "target_column": "Nationaliteit",
        "expected_categories": {"Totaal", "België", "EU15", "EU28", "niet-EU"},
        "prefer_group": "Werkend",
    },
    {
        "key": "resident_origin",
        "view": "BNW - T6ab (tabel)",
        "target_column": "Herkomst",
        "expected_categories": {"Totaal", "België", "EU15", "EU28", "Niet-EU"},
        "prefer_group": None,
    },
    {
        "key": "resident_generation",
        "view": "BNW - T6cd (tabel)",
        "target_column": "Generatie",
        "expected_categories": {"Totaal", "Geen vreemde herkomst", "Eerste", "Tweede", "Derde"},
        "prefer_group": "Werkend",
    },
]

SUPPLEMENT_META = OUTPUT_DIR / "metadata" / "supplemental_heterogeneity_plan.jsonl"
SUPPLEMENT_MANIFEST = OUTPUT_DIR / "metadata" / "supplemental_heterogeneity_manifest.json"
SUPPLEMENT_STATUS_JSON = OUTPUT_DIR / "reports" / "supplemental_heterogeneity_status.json"
SUPPLEMENT_STATUS_CSV = OUTPUT_DIR / "reports" / "supplemental_heterogeneity_status.csv"
SUPPLEMENT_PROBE_CSV = OUTPUT_DIR / "reports" / "supplemental_heterogeneity_probe_audit.csv"

def _supp_norm(x):
    return normal("" if x is None else x)

def _supp_param(params, names):
    wanted={_supp_norm(x) for x in names}
    for k,v in (params or {}).items():
        if _supp_norm(k) in wanted:
            return v
    for k,v in (params or {}).items():
        nk=_supp_norm(k)
        if any(w and w in nk for w in wanted):
            return v
    return None

def _supp_measure_is_worker_count(x):
    z=_supp_norm(x)
    return (
        "aantal" in z
        and "werkend" in z
        and "werkloos" not in z
        and "niet-beroepsact" not in z
        and "loontrekk" not in z
    )

def _supp_open_record(record):
    raw_rel=record.get("raw_file")
    if not raw_rel:
        raise ValueError("record has no raw_file")
    path=OUTPUT_DIR / raw_rel
    if not path.exists():
        raise FileNotFoundError(path)
    enc=record.get("encoding") or "utf-8-sig"
    sep=record.get("delimiter") or ","
    f=gzip.open(path,"rt",encoding=enc,errors="replace",newline="") if path.suffix.lower()==".gz" else open(
        path,"rt",encoding=enc,errors="replace",newline=""
    )
    return f

def _supp_inspect(record, spec, expected_year=None):
    """
    Inspect the raw CSV itself. A request is usable only if it contains a
    worker-count measure and the target categories. No rate is accepted as a
    substitute for counts.
    """
    out={
        "usable":False,
        "measure_label":None,
        "rows_worker_measure":0,
        "municipalities":0,
        "categories":[],
        "sex_values":[],
        "age_values":[],
        "year_values":[],
        "reason":None,
    }
    if not record or record.get("status")!="received" or not record.get("rows",0):
        out["reason"]="not_received_or_empty"
        return out

    try:
        with _supp_open_record(record) as f:
            reader=csv.DictReader(f,delimiter=record.get("delimiter") or ",")
            headers=reader.fieldnames or []
            if "Measure Names" not in headers or "Measure Values" not in headers:
                out["reason"]="measure_columns_missing"
                return out
            if spec["target_column"] not in headers:
                out["reason"]="target_column_missing"
                return out

            geo_col=next((c for c in ("Regio","Gemeente","Gemeente_titel","NIS") if c in headers),None)
            measures=set()
            categories=set()
            municipalities=set()
            sex_values=set()
            age_values=set()
            year_values=set()
            worker_rows=0

            for row in reader:
                m=(row.get("Measure Names") or "").strip()
                if m:
                    measures.add(m)
                if not _supp_measure_is_worker_count(m):
                    continue

                worker_rows += 1
                categories.add((row.get(spec["target_column"]) or "").strip())
                if geo_col:
                    municipalities.add((row.get(geo_col) or "").strip())
                if "Geslacht" in headers:
                    sex_values.add((row.get("Geslacht") or "").strip())
                if "Leeftijdsklasse" in headers:
                    age_values.add((row.get("Leeftijdsklasse") or "").strip())
                if "Jaar" in headers:
                    year_values.add((row.get("Jaar") or "").strip())

            worker_labels=[m for m in measures if _supp_measure_is_worker_count(m)]
            out["measure_label"]=worker_labels[0] if len(worker_labels)==1 else (
                worker_labels if worker_labels else None
            )
            out["rows_worker_measure"]=worker_rows
            out["municipalities"]=len({x for x in municipalities if x})
            out["categories"]=sorted(x for x in categories if x)
            out["sex_values"]=sorted(x for x in sex_values if x)
            out["age_values"]=sorted(x for x in age_values if x)
            out["year_values"]=sorted(x for x in year_values if x)

            missing_categories=spec["expected_categories"]-categories
            if not worker_labels:
                out["reason"]="worker_count_measure_absent"
            elif len(worker_labels)!=1:
                out["reason"]="multiple_worker_count_measures"
            elif missing_categories:
                out["reason"]="missing_categories:"+",".join(sorted(missing_categories))
            elif expected_year is not None and year_values and str(expected_year) not in year_values:
                out["reason"]="wrong_year"
            elif age_values and SUPPLEMENT_AGE not in age_values:
                out["reason"]="wrong_age"
            elif out["municipalities"] and out["municipalities"] < 500:
                out["reason"]="municipality_coverage_below_500"
            else:
                out["usable"]=True
                out["reason"]="ok"
    except Exception as exc:
        out["reason"]=f"inspect_error:{type(exc).__name__}:{exc}"
    return out

def _supp_all_request_meta():
    request_dir=OUTPUT_DIR / "metadata" / "requests"
    rows=[]
    for path in request_dir.glob("*.json"):
        try:
            m=read_json(path)
        except Exception:
            continue
        if m.get("status")=="received":
            rows.append(m)
    return rows

def _supp_existing_candidate(records, spec, year):
    """
    Reuse any already-downloaded request before making a new network call.
    """
    candidates=[]
    for rec in records:
        if _supp_norm(rec.get("view")) != _supp_norm(spec["view"]):
            continue
        p=rec.get("requested_parameters") or {}
        py=_supp_param(p,("Jaar","Year"))
        pa=_supp_param(p,("Leeftijdsklasse","Age"))
        if str(py).strip()!=str(year):
            continue
        if pa is not None and str(pa).strip() and _supp_norm(pa)!=_supp_norm(SUPPLEMENT_AGE):
            continue
        candidates.append(rec)

    # Prefer smaller responses first; they are more likely to already be
    # restricted to the requested sex/age scope.
    candidates=sorted(candidates,key=lambda r:(r.get("rows") or 10**18,r.get("request_id","")))
    for rec in candidates:
        audit=_supp_inspect(rec,spec,year)
        if audit["usable"]:
            return rec,audit
    return None,None

def _supp_reference_task(spec):
    """
    Use the verified final plan to inherit the source's working filter syntax,
    then override only the scope needed for this supplemental collection.
    """
    candidates=[]
    for task in iter_plan():
        if _supp_norm(task["view"]["name"]) != _supp_norm(spec["view"]):
            continue
        p=task.get("parameters") or {}
        py=_supp_param(p,("Jaar","Year"))
        pa=_supp_param(p,("Leeftijdsklasse","Age"))
        if str(py).strip()!="2019":
            continue
        if pa is not None and str(pa).strip() and _supp_norm(pa)!=_supp_norm(SUPPLEMENT_AGE):
            continue

        score=0
        group=_supp_param(p,("Groep",))
        if spec.get("prefer_group"):
            if group is None or str(group).strip()=="":
                score += 1
            elif _supp_norm(group)!=_supp_norm(spec["prefer_group"]):
                score += 100

        target=_supp_param(p,(spec["target_column"],))
        if target is not None and str(target).strip()!="":
            score += 50
        candidates.append((score,task))

    if not candidates:
        raise RuntimeError(f"No 2019 / {SUPPLEMENT_AGE} final-plan reference for {spec['view']}")
    candidates.sort(key=lambda x:(x[0],x[1]["request_id"]))
    return candidates[0][1]

def _supp_base_params(reference_task, spec, year):
    p=dict(reference_task.get("parameters") or {})
    # Keep the verified request structure but enforce the exact intended scope.
    p["Jaar"]=str(year)
    p["Leeftijdsklasse"]=SUPPLEMENT_AGE
    if "Geografisch niveau" in p:
        p["Geografisch niveau"]="Gemeente"
    if "Geslacht" in p:
        p["Geslacht"]="Totaal"
    if spec.get("prefer_group") and "Groep" in p:
        p["Groep"]=spec["prefer_group"]
    if spec["target_column"] in p:
        p[spec["target_column"]]=""
    return p

def _supp_probe_indicator(view, base_params, spec):
    """
    Try only a small, explicit set of Tableau parameter encodings.
    A variant is accepted solely when the returned raw CSV actually contains a
    worker-count Measure Name and the expected demographic categories.
    """
    variants=[
        ("Indicator_t5","6"),
        ("Parameters.Indicator_t5","6"),
        ("Indicator_t5","Aantal werkenden (n)"),
        ("Parameters.Indicator_t5","Aantal werkenden (n)"),
    ]
    audit_rows=[]
    for field,value in variants:
        params=dict(base_params)
        params[field]=value
        rec=CLIENT.fetch(view,params,"supplemental_heterogeneity_probe")
        chk=_supp_inspect(rec,spec,expected_year=2019)
        audit_rows.append({
            "key":spec["key"],
            "view":spec["view"],
            "indicator_field":field,
            "indicator_value":value,
            "request_id":rec.get("request_id"),
            "status":rec.get("status"),
            "rows":rec.get("rows"),
            "usable":chk["usable"],
            "measure_label":chk["measure_label"],
            "municipalities":chk["municipalities"],
            "categories":" | ".join(chk["categories"]),
            "reason":chk["reason"],
        })
        if chk["usable"]:
            return field,value,rec,chk,audit_rows
    return None,None,None,None,audit_rows

# ------------------------------------------------------------------
# 1. Build view map and scan the current cache.
# ------------------------------------------------------------------
prepared_by_name={_supp_norm(p["name"]):p for p in PREPARED}
existing_records=_supp_all_request_meta()

supplement_plan=[]
supplement_status=[]
probe_audit=[]
resolved_indicators={}

print("\n" + "="*92)
print("SUPPLEMENTAL HETEROGENEITY CHECK / CONTINUATION")
print("="*92)
print("Target years:",SUPPLEMENT_YEARS)
print("Age:",SUPPLEMENT_AGE)
print("Existing received request metadata:",f"{len(existing_records):,}")

for spec in SUPPLEMENT_SPECS:
    if _supp_norm(spec["view"]) not in prepared_by_name:
        raise KeyError(f"Prepared view missing: {spec['view']}")
    view=prepared_by_name[_supp_norm(spec["view"])]
    reference=_supp_reference_task(spec)

    print(f"\n[{spec['key']}] {spec['view']}")
    print("  Reference request:",reference["request_id"])

    # First determine whether every requested year is already available.
    year_hits={}
    for year in SUPPLEMENT_YEARS:
        rec,chk=_supp_existing_candidate(existing_records,spec,year)
        if rec is not None:
            year_hits[year]=(rec,chk)
            print(f"  {year}: reuse {rec['request_id']} | {chk['measure_label']} | municipalities={chk['municipalities']}")

    missing=[y for y in SUPPLEMENT_YEARS if y not in year_hits]

    # Probe a working indicator encoding only if at least one year is missing.
    indicator_field=None
    indicator_value=None
    if missing:
        base2019=_supp_base_params(reference,spec,2019)
        field,value,probe_rec,probe_chk,rows=_supp_probe_indicator(
            view,base2019,spec
        )
        probe_audit.extend(rows)
        if field is None:
            write_report(SUPPLEMENT_PROBE_CSV,probe_audit)
            raise RuntimeError(
                f"{spec['key']}: existing cache lacks worker-count data and "
                "no tested Indicator_t5 encoding produced a valid "
                "'Aantal werkenden (n)' response. See "
                f"{SUPPLEMENT_PROBE_CSV}"
            )
        indicator_field,indicator_value=field,value
        resolved_indicators[spec["key"]]={
            "field":field,
            "value":value,
            "measure_label":probe_chk["measure_label"],
        }
        print(
            "  Resolved indicator:",
            f"{field}={value!r}",
            "| measure=",probe_chk["measure_label"]
        )

        # Probe response itself may already satisfy 2019.
        if 2019 in missing and probe_chk["usable"]:
            year_hits[2019]=(probe_rec,probe_chk)

    # ------------------------------------------------------------------
    # 2. Continue only the still-missing years.
    # ------------------------------------------------------------------
    for year in SUPPLEMENT_YEARS:
        if year in year_hits:
            rec,chk=year_hits[year]
            source="cache"
        else:
            params=_supp_base_params(reference,spec,year)
            params[indicator_field]=indicator_value
            rec=CLIENT.fetch(
                view,params,"supplemental_heterogeneity_collection"
            )
            chk=_supp_inspect(rec,spec,expected_year=year)
            source="download_or_cache_by_url"

        status="received_valid" if chk["usable"] else "invalid"
        row={
            "key":spec["key"],
            "view":spec["view"],
            "year":year,
            "age":SUPPLEMENT_AGE,
            "request_id":rec.get("request_id") if rec else None,
            "status":status,
            "source":source,
            "rows":rec.get("rows") if rec else None,
            "raw_file":rec.get("raw_file") if rec else None,
            "measure_label":chk["measure_label"] if chk else None,
            "municipalities":chk["municipalities"] if chk else None,
            "categories":" | ".join(chk["categories"]) if chk else "",
            "reason":chk["reason"] if chk else "no_check",
            "requested_parameters":rec.get("requested_parameters") if rec else None,
        }
        supplement_status.append(row)

        if not chk["usable"]:
            write_json(SUPPLEMENT_STATUS_JSON,{str(i):r for i,r in enumerate(supplement_status)})
            write_report(
                SUPPLEMENT_STATUS_CSV,
                [{k:(jd(v) if isinstance(v,(dict,list)) else v) for k,v in r.items()}
                 for r in supplement_status]
            )
            raise RuntimeError(
                f"Supplemental request failed validation: "
                f"{spec['key']} {year} | {chk['reason']}"
            )

        supplement_plan.append({
            "supplement":SUPPLEMENT_NAME,
            "key":spec["key"],
            "view":{
                "name":view["name"],
                "module":view["module"],
                "csv_url":view["csv_url"],
            },
            "year":year,
            "age":SUPPLEMENT_AGE,
            "target_column":spec["target_column"],
            "measure_label":chk["measure_label"],
            "request_id":rec["request_id"],
            "parameters":rec.get("requested_parameters") or {},
            "raw_file":rec.get("raw_file"),
            "rows":rec.get("rows"),
            "validated_utc":utc_now(),
        })
        print(
            f"  {year}: {status} | rows={rec.get('rows')} | "
            f"municipalities={chk['municipalities']} | request={rec['request_id']}"
        )

# ------------------------------------------------------------------
# 3. Persist a separate supplemental plan and audit.
# ------------------------------------------------------------------
SUPPLEMENT_META.parent.mkdir(parents=True,exist_ok=True)
with SUPPLEMENT_META.open("w",encoding="utf-8") as f:
    for row in supplement_plan:
        f.write(jd(row)+"\n")

write_json(
    SUPPLEMENT_MANIFEST,
    {
        "supplement":SUPPLEMENT_NAME,
        "created_utc":utc_now(),
        "task_count":len(supplement_plan),
        "expected_task_count":len(SUPPLEMENT_SPECS)*len(SUPPLEMENT_YEARS),
        "years":SUPPLEMENT_YEARS,
        "age":SUPPLEMENT_AGE,
        "resolved_indicators":resolved_indicators,
        "scope_note":(
            "Supplement only. The original metadata/plan.jsonl and "
            "reports/task_status.json remain unchanged."
        ),
    }
)

write_json(
    SUPPLEMENT_STATUS_JSON,
    {str(i):r for i,r in enumerate(supplement_status)}
)
write_report(
    SUPPLEMENT_STATUS_CSV,
    [{k:(jd(v) if isinstance(v,(dict,list)) else v) for k,v in r.items()}
     for r in supplement_status]
)
write_report(SUPPLEMENT_PROBE_CSV,probe_audit)

# Hard completion check.
expected=len(SUPPLEMENT_SPECS)*len(SUPPLEMENT_YEARS)
valid=sum(r["status"]=="received_valid" for r in supplement_status)
if len(supplement_plan)!=expected or valid!=expected:
    raise RuntimeError(
        f"Supplement incomplete: plan={len(supplement_plan)}/{expected}, "
        f"valid={valid}/{expected}"
    )

print("\n" + "="*92)
print("SUPPLEMENT COMPLETE")
print("="*92)
print("Validated tasks:",valid,"/",expected)
print("Supplement plan:",SUPPLEMENT_META)
print("Manifest:",SUPPLEMENT_MANIFEST)
print("Status CSV:",SUPPLEMENT_STATUS_CSV)
print("Probe audit:",SUPPLEMENT_PROBE_CSV)
print(
    "\nThe original 7,118-task final plan was NOT modified. "
    "Re-running this cell is resumable: CLIENT.fetch reuses valid cached URLs."
)


## 10. 打包
保留原始响应和范围报告。所有计划请求完成，不等于全部VAR内部工作表及所有维度组合均已取得。

In [11]:
ARCHIVE = archive_results(include_raw=True)

输出压缩包： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_CSV_for_synthesis_20260922_124841.zip


In [12]:
from pathlib import Path
import zipfile
import json
from collections import defaultdict

ROOT = OUTPUT_DIR
OUT = ROOT.parent / "VAR_synthesis_handoff_small.zip"

# 每个视图最多放若干代表性非空原始文件，
# 用于检查字段结构，而不是用于正式合成。
SAMPLES_PER_VIEW = 3

request_dir = ROOT / "metadata" / "requests"
raw_root = ROOT / "raw"

samples = defaultdict(list)

for meta_file in request_dir.glob("*.json"):
    try:
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
    except Exception:
        continue

    if meta.get("status") != "received":
        continue
    if not meta.get("rows", 0):
        continue

    view = meta.get("view", "UNKNOWN")

    if len(samples[view]) >= SAMPLES_PER_VIEW:
        continue

    raw_rel = meta.get("raw_file")
    if not raw_rel:
        continue

    raw_path = ROOT / raw_rel
    if raw_path.exists():
        samples[view].append((meta_file, raw_path))

with zipfile.ZipFile(
    OUT,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as z:

    # 1. 所有元数据和报告
    for folder_name in ["metadata", "reports"]:
        folder = ROOT / folder_name
        for p in folder.rglob("*"):
            if p.is_file() and not p.name.endswith(".part"):
                z.write(
                    p,
                    p.relative_to(ROOT).as_posix()
                )

    # 2. 每个视图少量代表性 raw CSV
    for view, pairs in samples.items():
        for meta_file, raw_path in pairs:
            z.write(
                raw_path,
                "sample_raw/" + raw_path.relative_to(raw_root).as_posix()
            )

print("生成：", OUT)
print("文件大小：", f"{OUT.stat().st_size / 1024**2:.1f} MB")
print("包含视图数：", len(samples))
for view, pairs in sorted(samples.items()):
    print(view, ":", len(pairs), "sample files")

生成： D:\OneDrive - Universiteit Utrecht\31_BL_Network\raw_data\VAR_synthesis_handoff_small.zip
文件大小： 110.8 MB
包含视图数： 44
BNW - T1 (kaart) : 3 sample files
BNW - T1 (tabel) : 3 sample files
BNW - T2 (kaart) : 3 sample files
BNW - T2 (tabel) : 3 sample files
BNW - T3 (kaart) : 3 sample files
BNW - T3 (tabel) : 3 sample files
BNW - T4 (kaart) : 3 sample files
BNW - T4 (tabel) : 3 sample files
BNW - T5ab (kaart) : 3 sample files
BNW - T5ab (tabel) : 3 sample files
BNW - T5cd (kaart) : 3 sample files
BNW - T5cd (tabel) : 3 sample files
BNW - T6ab (kaart)  : 3 sample files
BNW - T6ab (tabel)  : 3 sample files
BNW - T6cd (kaart) : 3 sample files
BNW - T6cd (tabel) : 3 sample files
BW - JV : 1 sample files
BW - JV (tabel) : 1 sample files
BW - T1 (kaart) : 3 sample files
BW - T1 (tabel) : 1 sample files
BW - T2a (kaart) : 1 sample files
BW - T2a (tabel) : 3 sample files
BW - T2b (kaart) : 3 sample files
BW - T2b (tabel) : 3 sample files
BW - T3a herk (kaart) : 3 sample files
BW - T3a herk (tabel